In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

K = 100
TOP_M = 10
TRAIN_MIN_YEAR = 1964
TRAIN_MAX_YEAR = 1993
TEST_MIN_YEAR = 1994
TEST_MAX_YEAR = 2016

BASE_DIR = os.getcwd()
DATA_DIR = "data"

feature_cols = ["LME", "BEME", "r12_2", "OP", "Investment","ST_Rev", "LT_Rev", "AC", "IdioVol", "LTurnover"]

def load_all(data_dir: str):
    csv_files = sorted(glob.glob(os.path.join(data_dir, "y*.csv")))
    dfs = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        if not re.match(r"y\d{4}\.csv", fname):
            continue
        dfs.append(pd.read_csv(fpath))
    if not dfs:
        raise ValueError("no valid CSV files found in the specified directory.")
    return pd.concat(dfs, ignore_index=True)

df_all = load_all(DATA_DIR)
train_mask = (df_all["yy"] >= TRAIN_MIN_YEAR) & (df_all["yy"] <= TRAIN_MAX_YEAR)
df_train = df_all.loc[train_mask].copy()
if df_train.empty:
    raise ValueError("no rows found in the selected training interval.")

X_train = df_train
y_train = df_train["ret"]

test_mask = (df_all["yy"] >= TEST_MIN_YEAR) & (df_all["yy"] <= TEST_MAX_YEAR)
df_test = df_all.loc[test_mask].copy().reset_index(drop=True)
if df_test.empty:
    raise ValueError("no rows found in the selected test interval.")  

X_test = df_test
y_test = df_test["ret"]

In [ ]:
import pickle
from SR import compute_single_node_sr


top_n = 50
N_ESTIMATORS = 200
MAX_DEPTH = 4
MIN_SAMPLES_LEAF = 20
MAX_FEATURES = "sqrt"  # sqrt, log2 or None
RANDOM_STATE = 23

OUT_DIR2 = os.path.join(
    BASE_DIR,
    f"results_ne{N_ESTIMATORS}_md{MAX_DEPTH}_mf{MAX_FEATURES}"
)

save_path = os.path.join(OUT_DIR2, "top_items_by_row_by_k.pkl")
if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        top_items_by_row_by_k = pickle.load(f)

fig, ax = plt.subplots(figsize=(22, 7))

bg_color = "#EBEBEB"
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

rank_base = None

sr_pairs = []
excess_path = os.path.join(OUT_DIR2, f"EXCESS_ret_k{K}.csv")
df_excess = pd.read_csv(excess_path)
numeric_cols = df_excess.select_dtypes(include=["number"]).columns

for col in numeric_cols:
    sr = compute_single_node_sr(
        nodes_excess_path=excess_path,
        column=col,
    )
    sr_pairs.append((col, sr))

max_row_id, max_sr = max(sr_pairs, key=lambda x: x[1])
try:
    max_row_id_key = int(max_row_id)
except (TypeError, ValueError):
    max_row_id_key = max_row_id

top_pairs = sorted(sr_pairs, key=lambda x: x[1], reverse=True)[:top_n]

top_anchor_ids = pd.to_numeric([i for i, _ in top_pairs], errors="coerce")
top_anchor_ids = [int(i) for i in top_anchor_ids if pd.notna(i)]
top_anchor_ids = top_anchor_ids[:TOP_M]

valid_top_anchor_ids = [i for i in top_anchor_ids if 0 <= i <= (len(X_test) - 1)]
top_anchor_permnos = X_test.iloc[valid_top_anchor_ids]["permno"].tolist()
print(f"Top-{TOP_M} anchor_id -> permno: {list(zip(valid_top_anchor_ids, top_anchor_permnos))}")

if K not in top_items_by_row_by_k:
    available_k = sorted(top_items_by_row_by_k.keys()) if top_items_by_row_by_k else []
    print(f"Warnung: K={K} fehlt in {save_path}. Verfuegbare K-Werte: {available_k}.")
    top_items_for_k = {}
else:
    top_items_for_k = top_items_by_row_by_k[K]

for anchor_id in top_anchor_ids:
    top_items = top_items_for_k.get(anchor_id, {})
    if not top_items:
        print(f"Hinweis: Keine Nachbarn fuer anchor_id={anchor_id} gefunden.")
        continue

    rows = []
    for month, items in top_items.items():
        for item_id, weight in items:
            rows.append({
                "month": month,
                "item_id": item_id,
                "weight": weight,
            })

    neighbors_df = pd.DataFrame(rows)
    if not neighbors_df.empty:
        neighbors_df["item_id"] = pd.to_numeric(neighbors_df["item_id"], errors="coerce")
        neighbors_df = neighbors_df.dropna(subset=["item_id"]).copy()
        neighbors_df["item_id"] = neighbors_df["item_id"].astype(int)

        max_index = len(X_test) - 1
        valid_mask = (neighbors_df["item_id"] >= 0) & (neighbors_df["item_id"] <= max_index)
        neighbors_df = neighbors_df.loc[valid_mask].copy()

        neighbor_full_rows = X_test.iloc[neighbors_df["item_id"].to_numpy()].reset_index(drop=True)
        neighbors_df = pd.concat([neighbors_df.reset_index(drop=True), neighbor_full_rows], axis=1)

    neighbors_path = os.path.join(OUT_DIR2, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    neighbors_df.to_csv(neighbors_path, index=False)
    print(f"Nachbarn inkl. X_test-Spalten gespeichert unter: {neighbors_path}")

print(f"k={K} | max SR row_id={max_row_id} | SR={max_sr:.4f}")
print("-")

best_idx = int(max_row_id_key)
print(X_test.iloc[[best_idx]][["permno"]])

sr_pairs_sorted = sorted(top_pairs, key=lambda x: x[1], reverse=True)
sorted_i = [i for i, _ in sr_pairs_sorted]
sorted_sr = [sr for _, sr in sr_pairs_sorted]
rank = list(range(1, len(sorted_sr) + 1))

top50_df = pd.DataFrame({
    "rank": rank,
    "anchor_id": pd.to_numeric(sorted_i, errors="coerce"),
    "sr": sorted_sr,
})
top50_df = top50_df.dropna(subset=["anchor_id"]).copy()
top50_df["anchor_id"] = top50_df["anchor_id"].astype(int)
max_index = len(X_test) - 1
top50_df = top50_df[(top50_df["anchor_id"] >= 0) & (top50_df["anchor_id"] <= max_index)].copy()

full_rows_df = X_test.iloc[top50_df["anchor_id"].to_numpy()].reset_index(drop=True)
result_df = pd.concat(
    [top50_df[["rank", "anchor_id", "sr"]].reset_index(drop=True), full_rows_df],
    axis=1,
 )

if "permno" not in result_df.columns:
    raise KeyError(f"column 'permno' missing in result_df. Available columns: {result_df.columns.tolist()}")

file_path = os.path.join(OUT_DIR2, f"top50_anchor_fullrows_k{K}.csv")
result_df.to_csv(file_path, index=False)

if rank_base is None:
    rank_base = rank

if "LME" in result_df.columns:
    lme_vals = pd.to_numeric(result_df["LME"], errors="coerce")
    finite_lme = lme_vals.dropna()

    if not finite_lme.empty:
        norm = plt.Normalize(vmin=0.0, vmax=0.5, clip=True)
        bar_colors = [plt.cm.Blues(0.25 + 0.70 * norm(v)) if pd.notna(v) else (0.7, 0.7, 0.7, 1.0) for v in lme_vals]
        sm = plt.cm.ScalarMappable(cmap="Blues", norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.01)
        cbar.set_label("LME")
        cbar.set_ticks([0, 0.25, 0.5])
    else:
        bar_colors = "#6C757D"

    ax.bar(
        rank,
        sorted_sr,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.5,
        width=0.82,
        label=f"SR (k = {K})",
    )
else:
    ax.bar(
        rank,
        sorted_sr,
        color="#20B2B8",
        edgecolor="white",
        linewidth=0.5,
        width=0.82,
        label=f"SR (k = {K})",
    )

ax.set_title(f"n_estimators = {N_ESTIMATORS}; k = {K}; depth = {MAX_DEPTH}", pad=18)
ax.set_xlabel("Rank (1 = highest SR)")
ax.set_ylabel("Monthly Sharpe Ratio (SR)")

if rank_base is not None:
    ax.set_xticks(rank_base)
    ax.set_xticklabels([str(r) for r in rank_base])
    ax.set_xlim(0.5, len(rank_base) + 0.5)

ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
def load_factor_matrix(factor_path: str) -> pd.DataFrame:
    factor_file = os.path.join(factor_path, "tradable_factors.csv")
    return pd.read_csv(factor_file).iloc[360:636, :].reset_index(drop=True)


def get_factors(factor_mat: pd.DataFrame, model: str) -> np.ndarray:
    if model == "FF3":
        return factor_mat.iloc[:, 1:4].to_numpy()
    if model == "FF5":
        return factor_mat.iloc[:, [1, 2, 3, 5, 6]].to_numpy()
    if model == "FF11":
        return factor_mat.iloc[:, 1:12].to_numpy()
    raise ValueError(f"Unknown model: {model}")


def alpha_tstat(y: np.ndarray, factors: np.ndarray) -> tuple[float, float]:
    y = np.asarray(y, dtype=float).reshape(-1)
    factors = np.asarray(factors, dtype=float)

    X = np.column_stack([np.ones(factors.shape[0]), factors])
    n, k = X.shape
    if n <= k:
        raise ValueError(f"Not enough observations: n={n}, k={k}")

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta

    sigma2 = float(resid.T @ resid) / (n - k)
    xtx_inv = np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(sigma2 * xtx_inv))
    if se[0] == 0:
        raise ValueError("Intercept standard error is zero; t-stat undefined.")

    alpha = float(beta[0])
    t_stat = float(alpha / se[0])
    return alpha, t_stat


def load_anchor_info(top50_path: str) -> pd.DataFrame:
    if not os.path.exists(top50_path):
        raise FileNotFoundError(f"{top50_path} missing")

    top50_df = pd.read_csv(top50_path)
    required_cols = {"anchor_id", "permno"}
    missing = required_cols - set(top50_df.columns)
    if missing:
        raise KeyError(f"{top50_path}: {missing} column missing")

    top50_df["anchor_id"] = pd.to_numeric(top50_df["anchor_id"], errors="coerce")
    top50_df["permno"] = pd.to_numeric(top50_df["permno"], errors="coerce")
    top50_df = top50_df.dropna(subset=["anchor_id", "permno"]).copy()
    top50_df["anchor_id"] = top50_df["anchor_id"].astype(int)
    top50_df["permno"] = top50_df["permno"].astype(int)

    return top50_df[["anchor_id", "permno"]].drop_duplicates(subset=["anchor_id"], keep="first").reset_index(drop=True)

FACTOR_PATH = "PATH"
FF_MODELS = ["FF3", "FF5", "FF11"]
MODEL_COLORS = {"FF3": "#A7C7E7", "FF5": "#BFD67A", "FF11": "#D8CCF2"}

excess_path = os.path.join(OUT_DIR2, f"EXCESS_ret_k{K}.csv")
top50_path = os.path.join(OUT_DIR2, f"top50_anchor_fullrows_k{K}.csv")

df_excess = pd.read_csv(excess_path)
anchor_info = load_anchor_info(top50_path)
factor_mat = load_factor_matrix(FACTOR_PATH)

top50_df = pd.read_csv(top50_path)
required_cols = {"anchor_id", "sr", "permno"}
missing_cols = required_cols - set(top50_df.columns)
if missing_cols:
    raise KeyError(f"Fehlende Spalten in {top50_path}: {missing_cols}")

top50_df["anchor_id"] = pd.to_numeric(top50_df["anchor_id"], errors="coerce")
top50_df["sr"] = pd.to_numeric(top50_df["sr"], errors="coerce")
top50_df["permno"] = pd.to_numeric(top50_df["permno"], errors="coerce")
top50_df = top50_df.dropna(subset=["anchor_id", "sr", "permno"]).copy()
top50_df["anchor_id"] = top50_df["anchor_id"].astype(int)
top50_df["permno"] = top50_df["permno"].astype(int)

top_sr_df = (
    top50_df.sort_values("sr", ascending=False)
    .drop_duplicates(subset=["anchor_id"])
    .head(TOP_M)
    .reset_index(drop=True)
)

selected_permnos = top_sr_df["permno"].astype(int).tolist()
print(f"Top-{TOP_M} permno nach SR: {selected_permnos}")

anchor_cols = top_sr_df["anchor_id"].astype(str).tolist()
rank = list(range(1, len(anchor_cols) + 1))

model_to_tstats: dict[str, list[float]] = {}
anchor_model_stats: dict[str, dict[str, float]] = {
    str(int(row.anchor_id)): {"sr": float(row.sr)}
    for _, row in top_sr_df.iterrows()
}

for model in FF_MODELS:
    factors = get_factors(factor_mat, model)
    t_stats = []

    for col in anchor_cols:
        if col not in df_excess.columns:
            raise KeyError
        
        alpha, t = alpha_tstat(df_excess[col].to_numpy()[: factors.shape[0]], factors)
        t_stats.append(t)

        if col in anchor_model_stats:
            anchor_model_stats[col][f"alpha{model}"] = float(alpha)
            anchor_model_stats[col][f"tstat{model}"] = float(t)

    model_to_tstats[model] = t_stats

alpha_tstat_df_topm = pd.DataFrame([
    {
        "anchorid": int(aid),
        **vals,
    }
    for aid, vals in anchor_model_stats.items()
]).sort_values("sr", ascending=False).reset_index(drop=True)

alpha_tstat_display = alpha_tstat_df_topm.merge(
    top_sr_df[["anchor_id", "permno"]].rename(columns={"anchor_id": "anchorid"}),
    on="anchorid",
    how="left",
)

print(alpha_tstat_display[["permno", "sr", "alphaFF3", "tstatFF3", "alphaFF5", "tstatFF5", "alphaFF11", "tstatFF11"]])

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(
    nrows=len(FF_MODELS),
    ncols=1,
    figsize=(22, 12),
    sharex=True,
    gridspec_kw={"hspace": 0.08},
    constrained_layout=True,
)
if len(FF_MODELS) == 1:
    axes = [axes]

bg_color = "#EBEBEB"
fig.patch.set_facecolor(bg_color)

for ax, model in zip(axes, FF_MODELS):
    ax.set_facecolor(bg_color)
    vals = model_to_tstats[model]

    ax.bar(
        rank,
        vals,
        color=MODEL_COLORS[model],
        edgecolor="white",
        linewidth=0.5,
        width=0.85,
    )

    ax.axhline(1.96, color="#3B3B3B", linewidth=2.2, linestyle="--", zorder=4)
    ax.axhline(2.576, color="#1F1F1F", linewidth=2.6, linestyle="--", zorder=4)

    ax.set_title(model, loc="left", fontweight="bold")
    ax.set_xlim(0.5, len(rank) + 0.5)
    ax.set_ylim(bottom=0.0)
    ax.grid(False)

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

axes[-1].set_xlabel("Rank (1 = highest SR)")
axes[-1].set_xticks(rank)
axes[-1].set_xticklabels([str(r) for r in rank])
fig.supylabel("t-statistic of alpha")

fig.suptitle(f"n_estimators = {N_ESTIMATORS}; k = {K}; depth = {MAX_DEPTH}", fontsize=18)
plt.show()

In [ ]:
from weighted_characteristics_monthly import (
    compute_weighted_characteristics_by_month,
    default_output_path,
)


top50_path = os.path.join(OUT_DIR2, f"top50_anchor_fullrows_k{K}.csv")
if not os.path.exists(top50_path):
    raise FileNotFoundError(f"Datei nicht gefunden: {top50_path}. Bitte zuerst Zelle 2 ausfuehren.")

top50_df = pd.read_csv(top50_path)
for col in ["anchor_id", "permno", "sr"]:
    if col not in top50_df.columns:
        raise KeyError(f"Column '{col}' missing in {top50_path}. Available columns: {top50_df.columns.tolist()}")

top50_df["anchor_id"] = pd.to_numeric(top50_df["anchor_id"], errors="coerce")
top50_df["permno"] = pd.to_numeric(top50_df["permno"], errors="coerce")
top50_df["sr"] = pd.to_numeric(top50_df["sr"], errors="coerce")
top50_df = top50_df.dropna(subset=["anchor_id", "permno", "sr"]).copy()
top50_df["anchor_id"] = top50_df["anchor_id"].astype(int)
top50_df["permno"] = top50_df["permno"].astype(int)

top_anchor_meta = (
    top50_df.sort_values("sr", ascending=False)[["anchor_id", "permno", "sr"]]
    .drop_duplicates(subset=["anchor_id"])
    .head(TOP_M)
    .reset_index(drop=True)
)

if top_anchor_meta.empty:
    raise ValueError("No valid anchor metadata found in top50_df. Please check the contents of the CSV file.")

summary_rows = []
for row in top_anchor_meta.itertuples(index=False):
    anchor_id = int(row.anchor_id)
    input_csv = os.path.join(OUT_DIR2, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"File not found: {input_csv}")

    df = pd.read_csv(input_csv)
    try:
        out = compute_weighted_characteristics_by_month(df)
    except Exception as e:
        raise RuntimeError(f"Error occurred while processing anchor_id={anchor_id}: {e}")

    output_csv = default_output_path(input_csv)
    out.to_csv(output_csv, index=False)
    print(f"Saved weighted monthly characteristics to: {output_csv}")

    avg_lme = float("nan")
    if "LME" in out.columns:
        avg_lme = float(pd.to_numeric(out["LME"], errors="coerce").mean())
    else:
        print(f"Hinweis: LME fehlt in aggregierter Datei fuer anchor_id={anchor_id}.")

    summary_rows.append({
        "anchorid": anchor_id,
        "avglme": avg_lme,
    })

if summary_rows:
    lme_summary_df_topm = pd.DataFrame(summary_rows)
    lme_display_df = lme_summary_df_topm.merge(
        top_anchor_meta[["anchor_id", "permno"]].rename(columns={"anchor_id": "anchorid"}),
        on="anchorid",
        how="left",
    )
    lme_display_df = lme_display_df[["permno", "avglme"]].sort_values("avglme", ascending=False).reset_index(drop=True)
    print(lme_display_df)
else:
    lme_summary_df_topm = pd.DataFrame(columns=["anchorid", "avglme"])

In [ ]:
# ===================== TURNOVER CALCULATION FUNCTION =====================
def compute_turnover_for_anchor(df_top: pd.DataFrame, returns_df: pd.DataFrame) -> pd.DataFrame:
    df_top = df_top.copy()
    df_top["month_idx"] = df_top["yy"] * 100 + df_top["mm"]
    df_top = df_top.sort_values(["month_idx", "permno"]).reset_index(drop=True)

    returns_df = returns_df.copy()
    returns_df["month_idx"] = returns_df["yy"] * 100 + returns_df["mm"]

    months = sorted(df_top["month_idx"].unique())
    rows = []

    for i in range(len(months) - 1):
        month_prev = months[i]
        month_curr = months[i + 1]

        df_prev = df_top[df_top["month_idx"] == month_prev].copy()
        df_curr = df_top[df_top["month_idx"] == month_curr].copy()

        if df_curr.empty:
            continue

        w_prev = df_prev.set_index("permno")["weight"] if not df_prev.empty else pd.Series(dtype=float)
        w_curr = df_curr.set_index("permno")["weight"]

        ret_curr = returns_df[returns_df["month_idx"] == month_curr]
        r_curr = ret_curr.set_index("permno")["ret"] if not ret_curr.empty else pd.Series(dtype=float)

        prev_with_returns = w_prev.index.intersection(r_curr.index)
        if len(prev_with_returns) == 0:
            R_p_t = 0.0
        else:
            R_p_t = float((w_prev.loc[prev_with_returns] * r_curr.loc[prev_with_returns]).sum())

        missing_asset_returns = max(0, len(w_prev) - len(prev_with_returns))


        turnover_sum = 0.0
        all_perms = w_prev.index.union(w_curr.index)
        for perm in all_perms:
            w_t = float(w_curr.get(perm, 0.0))
            w_t_minus1 = float(w_prev.get(perm, 0.0))
            R_i_t = float(r_curr.get(perm, 0.0)) if perm in r_curr.index else 0.0

            denom = 1.0 + R_p_t
            if denom == 0.0:
                w_drift = w_t_minus1
            else:
                w_drift = w_t_minus1 * (1.0 + R_i_t) / denom

            turnover_sum += abs(w_t - w_drift)

        rows.append({
            "month_idx": month_curr,
            "turnover": float(turnover_sum),
            "R_p_t": float(R_p_t),
            "n_assets_prev": int(len(w_prev)),
            "n_assets_curr": int(len(w_curr)),
            "missing_asset_returns": int(missing_asset_returns),
        })

    return pd.DataFrame(rows)

try:
    top_anchor_ids = top_anchor_meta["anchor_id"].tolist()
except Exception:
    top_anchor_ids = []

print(f"Suche nach Top-Items Dateien fuer K={K}...")
print(f"Top Anchor IDs: {top_anchor_ids}")

required_return_cols = {"yy", "mm", "permno", "ret"}
missing_return_cols = required_return_cols - set(df_test.columns)
if missing_return_cols:
    raise KeyError(f"Fehlende Spalten in df_test fuer Returns: {missing_return_cols}")

returns_df = df_test[["yy", "mm", "permno", "ret"]].copy()

turnover_rows = []
for anchor_id in top_anchor_ids:
    top_items_path = os.path.join(OUT_DIR2, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    if not os.path.exists(top_items_path):
        print(f"Uebersprungen (Datei fehlt): {top_items_path}")
        continue

    df_top = pd.read_csv(top_items_path)

    try:
        turnover_df = compute_turnover_for_anchor(df_top, returns_df)
    except Exception as e:
        print(f"Uebersprungen (Fehler) fuer anchor_id={anchor_id}: {e}")
        import traceback
        traceback.print_exc()
        continue

    if turnover_df.empty:
        print(f"Keine Turnover-Zeilen fuer anchor_id={anchor_id} erzeugt.")
        continue

    avg_turnover = float(turnover_df["turnover"].mean())
    print(f"✓ anchor_id={anchor_id}: avg_turnover={avg_turnover:.6f} (n_months={len(turnover_df)})")

    turnover_rows.append({
        "anchorid": int(anchor_id),
        "avgturnover": avg_turnover,
    })

if turnover_rows:
    turnover_summary_df_topm = pd.DataFrame(turnover_rows)
    turnover_display_df = turnover_summary_df_topm.merge(
        top_anchor_meta[["anchor_id", "permno"]].rename(columns={"anchor_id": "anchorid"}),
        on="anchorid",
        how="left",
    )
    turnover_display_df = turnover_display_df[["permno", "avgturnover"]].sort_values("avgturnover", ascending=False).reset_index(drop=True)
    print(turnover_display_df)
else:
    turnover_summary_df_topm = pd.DataFrame(columns=["anchorid", "avgturnover"])


base_df = top_anchor_meta[["anchor_id", "permno", "sr"]].copy()
base_df = base_df.rename(columns={"anchor_id": "anchorid"})

alpha_cols = [c for c in ["anchorid", "alphaFF3", "tstatFF3", "alphaFF5", "tstatFF5", "alphaFF11", "tstatFF11"] if c in alpha_tstat_df_topm.columns]
combined_df = base_df.merge(alpha_tstat_df_topm[alpha_cols] if alpha_cols else alpha_tstat_df_topm, on="anchorid", how="left")

if not lme_summary_df_topm.empty:
    combined_df = combined_df.merge(
        lme_summary_df_topm,
        on="anchorid",
        how="left"
    )
else:
    combined_df["avglme"] = np.nan

if not turnover_summary_df_topm.empty:
    combined_df = combined_df.merge(
        turnover_summary_df_topm,
        on="anchorid",
        how="left"
    )
else:
    combined_df["avgturnover"] = np.nan

ordered_cols = [
    "anchorid", "permno", "sr",
    "alphaFF3", "tstatFF3", "alphaFF5", "tstatFF5", "alphaFF11", "tstatFF11",
    "avglme", "avgturnover"
]

ordered_cols = [col for col in ordered_cols if col in combined_df.columns]
combined_df = combined_df[ordered_cols]

combined_csv_path = os.path.join(OUT_DIR2, f"top{TOP_M}_anchor_metrics_combined_k{K}.csv")
combined_df.to_csv(combined_csv_path, index=False)
print(combined_df.head())